Link da fonte dos dados: https://www.kaggle.com/datasets/victorsoeiro/netflix-tv-shows-and-movies?select=titles.csv

In [ ]:
import pandas as pd
import ast

In [ ]:
df = pd.read_csv('titles.csv')
print(f"Linhas antes do tratamento: {len(df)}")

Linhas antes do tratamento: 5850


In [ ]:
df = df.drop_duplicates(subset='id') # remover duplicatas por id
df = df.dropna(subset=['title']) # remove linha com título nulo (apenas 1 caso)

In [ ]:
# converter colunas de lista (string -> lista real)
def parse_lista(valor):
  try:
    return ast.literal_eval(valor)
  except (ValueError, SyntaxError):
    return []

df['genres'] = df['genres'].apply(parse_lista)
df['production_countries'] = df['production_countries'].apply(parse_lista)

In [ ]:
# tratar age_certification (nulo --> "Não informado")
df['age_certification'] = df['age_certification'].fillna('Não informado')

In [ ]:
# criar coluna de década de lançamento
df['decada'] = (df['release_year'] // 10) * 10

In [ ]:
# tratar notas: manter nulos como estão (não inventar valor), mas criar flag de "dado completo" para relatórios que exigem nota
df['tem_nota_imdb'] = df['imdb_score'].notna()

In [ ]:
# criar flag "alta performance"
# alta performance = nota IMDB >= 7 E votos acima da mediana
# evita que filme com nota alta e poucos votos pareça sucesso
mediana_votos = df['imdb_votes'].median()
df['alta_performance'] = (
  (df['imdb_score'] >= 7) & (df['imdb_votes'] >= mediana_votos)
)

In [ ]:
# expandir gêneros (uma linha por gênero, para contagem correta)
df_generos = df.explode('genres').rename(columns={'genres': 'genero'})
df_generos = df_generos.dropna(subset=['genero'])
df_generos = df_generos[df_generos['genero'] != '']

In [ ]:
# expandir países de produção (uma linha por país)
df_paises = df.explode('production_countries').rename(
    columns={'production_countries': 'pais'}
)

df_paises = df_paises.dropna(subset=['pais'])
df_paises = df_paises[df_paises['pais'] != '']

In [ ]:
df_export = df.copy()
df_export['genres'] = df_export['genres'].apply(lambda x: ', '.join(x))
df_export['production_countries'] = df_export['production_countries'].apply(lambda X: ', '.join(X))

df_export.to_csv('netflix_tratado.csv', index=False, encoding='utf-8-sig')
df_generos[['id', 'title', 'genero']].to_csv(
    'netflix_generos.csv', index=False, encoding='utf-8-sig'
)

df_paises[['id', 'title', 'pais']].to_csv(
    'netflix_paises.csv', index=False, encoding='utf-8-sig'
)

print(f"Linhas depois do tratamento: {len(df_export)}")
print(f"Mediana de votos IMDB (usada no filtro de alta performance): {mediana_votos}")
print("\nArquivos gerados:")
print("- netflix_tratado.csv  (tabela fato principal)")
print("- netflix_generos.csv  (dimensão de gêneros, 1 linha por gênero)")
print("- netflix_paises.csv   (dimensão de países, 1 linha por país)")

Linhas depois do tratamento: 5849
Mediana de votos IMDB (usada no filtro de alta performance): 2233.5

Arquivos gerados:
- netflix_tratado.csv  (tabela fato principal)
- netflix_generos.csv  (dimensão de gêneros, 1 linha por gênero)
- netflix_paises.csv   (dimensão de países, 1 linha por país)
